In [ ]:
# --- Repository setup ---
import os
from pathlib import Path

# Find the repository root (the folder that contains data/graphs) and work from there,
# so every relative path below resolves no matter where Jupyter was started.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'graphs').is_dir())
os.chdir(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT)


# Build evaluation table from graph runs

This notebook scans `results/raw_runs/S1_symbolic/**` for per-case `__graph.json` files produced by `symtom_collection.ipynb`, extracts key evaluation metrics, displays them as a table, and exports a CSV.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

ROOT = Path('results/raw_runs/S1_symbolic')
if not ROOT.exists():
    raise FileNotFoundError(f'Graph eval root not found: {ROOT.resolve()}')

print('Scanning:', ROOT.resolve())

Scanning: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase2_eval_runs/graph


In [2]:
# Find all per-case run files (exclude aggregated *__graph_eval.json files)
graph_files = sorted(p for p in ROOT.rglob('*.json') if p.name.endswith('__graph.json'))
print('Found __graph.json files:', len(graph_files))

# Preview a few
for p in graph_files[:5]:
    rel = p.relative_to(ROOT)
    print('-', rel)

Found __graph.json files: 22
- CMH/CMH_single_hop_single_hop_negative_1__CMH_single_hop_negative_1__graph.json
- CMH/CMH_single_hop_single_hop_positive_1__CMH_single_hop_positive_1__graph.json
- CMH/CMH_to_SUI_multi_hop_multi_hop_positive_1__CMH_to_SUI_multi_hop_positive_1__graph.json
- DEP/DEP_single_hop_single_hop_negative_1__DEP_single_hop_negative_001__graph.json
- DEP/DEP_single_hop_single_hop_positive_1__DEP_single_hop_positive_001__graph.json


In [3]:
def safe_int(x: Any) -> int | None:
    if x is None:
        return None
    try:
        return int(x)
    except Exception:
        return None


def extract_row(path: Path) -> dict[str, Any]:
    data = json.loads(path.read_text(encoding='utf-8'))

    # Module folder is the first directory under ROOT (e.g., DEP, SUI)
    rel_parts = path.relative_to(ROOT).parts
    module_folder = rel_parts[0] if len(rel_parts) >= 2 else path.parent.name

    evaluation = data.get('evaluation') or {}
    counts = evaluation.get('counts') or {}

    # Prefer the root vp_total_tokens (present in saved __graph.json)
    vp_total_tokens = (
        safe_int(data.get('vp_total_tokens'))
        or safe_int(evaluation.get('vp_total_tokens'))
        or safe_int((((data.get('usage') or {}).get('vp') or {}).get('total_tokens')))
    )

    return {
        'file_name': path.name,
        'module_folder': module_folder,
        'expected_assessment_findings': safe_int(counts.get('expected_assessment_findings')),
        'matched': safe_int(counts.get('matched')),
        'missing': safe_int(counts.get('missing')),
        'mismatched': safe_int(counts.get('mismatched')),
        'extra': safe_int(counts.get('extra')),
        'vp_total_tokens': vp_total_tokens,
    }


rows = [extract_row(p) for p in graph_files]
print('Rows:', len(rows))

Rows: 22


In [4]:
# Display as a table
try:
    import pandas as pd

    df = pd.DataFrame(rows)
    df = df.sort_values(['module_folder', 'file_name']).reset_index(drop=True)
    df
except Exception as e:
    print('pandas not available; showing first 10 rows as dicts. Error:', str(e))
    for r in rows[:10]:
        print(r)

In [5]:
# Export CSV
OUTPUT_CSV = Path('results/summaries/S1_symbolic_summary.csv')

columns = [
    'file_name',
    'module_folder',
    'expected_assessment_findings',
    'matched',
    'missing',
    'mismatched',
    'extra',
    'vp_total_tokens',
]

try:
    import pandas as pd

    out_df = pd.DataFrame(rows)[columns].sort_values(['module_folder', 'file_name'])
    out_df.to_csv(OUTPUT_CSV, index=False)
    print('Saved CSV:', OUTPUT_CSV.resolve())
except Exception:
    import csv

    with OUTPUT_CSV.open('w', encoding='utf-8', newline='') as f:
        w = csv.DictWriter(f, fieldnames=columns)
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k) for k in columns})
    print('Saved CSV (csv module):', OUTPUT_CSV.resolve())

Saved CSV: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase2_eval_runs/graph_eval_summary.csv


In [9]:
out_df

,file_name,module_folder,expected_assessment_findings,matched,missing,mismatched,extra,vp_total_tokens
0,CMH_single_hop_single_hop_negative_1__CMH_sing...,CMH,11,11,0,0,0,7464
1,CMH_single_hop_single_hop_positive_1__CMH_sing...,CMH,3,3,0,0,6,5457
2,CMH_to_SUI_multi_hop_multi_hop_positive_1__CMH...,CMH,7,7,0,0,3,5674
3,DEP_single_hop_single_hop_negative_1__DEP_sing...,DEP,5,5,0,0,0,2385
4,DEP_single_hop_single_hop_positive_1__DEP_sing...,DEP,8,8,0,0,0,7488
5,DEP_to_CMH_multi_hop_multi_hop_positive_1__DEP...,DEP,23,23,0,0,0,13528
6,DEP_to_OTH_multi_hop_multi_hop_positive_1__DEP...,DEP,6,6,0,0,0,3000
7,DEP_to_PSY_multi_hop_multi_hop_positive_1__DEP...,DEP,10,10,0,0,0,6611
8,DEP_to_SUI_multi_hop_multi_hop_positive_1__DEP...,DEP,10,10,0,0,3,12009
9,EPI_single_hop_single_hop_negative_1__EPI_sing...,EPI,4,4,0,0,0,1813


In [7]:
# (Optional) quick module-level summary
try:
    import pandas as pd

    summary = (
        pd.DataFrame(rows)
        .groupby('module_folder', dropna=False)
        .agg(
            files=('file_name', 'count'),
            expected=('expected_assessment_findings', 'sum'),
            matched=('matched', 'sum'),
            missing=('missing', 'sum'),
            mismatched=('mismatched', 'sum'),
            extra=('extra', 'sum'),
            vp_total_tokens=('vp_total_tokens', 'sum'),
        )
        .reset_index()
        .sort_values('module_folder')
    )
    summary
except Exception:
    print('Skipping summary (pandas not available).')

In [8]:
summary

,module_folder,files,expected,matched,missing,mismatched,extra,vp_total_tokens
0,CMH,3,21,21,0,0,9,18595
1,DEP,6,62,62,0,0,3,45021
2,EPI,3,25,24,0,1,0,14814
3,PSY,3,21,19,0,2,0,12619
4,SUB,4,32,32,0,0,0,18056
5,SUI,3,22,22,0,0,0,11805
